In [ ]:
!pip -q install datasets

import re
import cupy as xp
from collections import Counter
from datasets import load_dataset

# --- download TinyStories and take a slice ---
N_STORIES = 80_000
ds = load_dataset("roneneldan/TinyStories", split="train")
stories = ds[:N_STORIES]["text"]
print(f"{len(stories):,} stories")

# --- tokenize: lowercase, words (apostrophes kept) or single punctuation ---
token_re = re.compile(r"[a-z']+|[^\sa-z']")
def tokenize(s):
    return token_re.findall(s.lower())

EOS = "<EOS>"   # marks story boundaries
UNK = "<UNK>"   # stand-in for out-of-vocab words

all_tokens = []
for s in stories:
    all_tokens.extend(tokenize(s))
    all_tokens.append(EOS)
print(f"{len(all_tokens):,} tokens in the stream")

# --- build the vocab: top VOCAB_SIZE most frequent tokens ---
VOCAB_SIZE = 3000
counts = Counter(all_tokens)
most_common = [tok for tok, _ in counts.most_common(VOCAB_SIZE - 2)]
vocab = [UNK, EOS] + [t for t in most_common if t not in (UNK, EOS)]
vocab = vocab[:VOCAB_SIZE]
stoi = {t: i for i, t in enumerate(vocab)}   # token -> id  (UNK=0, EOS=1)
itos = {i: t for t, i in stoi.items()}       # id -> token
vocab_size = len(vocab)

# --- coverage check: how much of the stream survives without <UNK>? ---
covered = sum(c for t, c in counts.items() if t in stoi)
print(f"vocab size: {vocab_size}, coverage: {100 * covered / len(all_tokens):.2f}%")

# --- tokens -> ids, straight to GPU ---
unk_id = stoi[UNK]
data = xp.array([stoi.get(t, unk_id) for t in all_tokens], dtype=xp.int64)

# --- sliding windows via the index-grid trick ---
BLOCK_SIZE = 16
n_windows = len(data) - BLOCK_SIZE
offsets = xp.arange(n_windows)[:, None] + xp.arange(BLOCK_SIZE)[None, :]
X = data[offsets]           # (n_windows, 16)
Y = data[BLOCK_SIZE:]       # (n_windows,)

# --- train / val split ---
n = int(0.9 * n_windows)
Xtr, Ytr = X[:n], Y[:n]
Xva, Yva = X[n:], Y[n:]
print(f"train windows: {len(Xtr):,}   val windows: {len(Xva):,}")

# --- sanity check: decode one window back to words ---
i = 54321
print([itos[int(j)] for j in X[i]], "->", itos[int(Y[i])])

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

data/train-00002-of-00004-a26307300439e9(…): reconstructing file:   0%|          |  0.00B /  246MB            

... (4 outputs left out)

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

80,000 stories
16,605,770 tokens in the stream
vocab size: 2999, coverage: 98.20%
train windows: 14,945,178   val windows: 1,660,576
['cave', '.', 'he', 'thought', 'it', 'would', 'be', 'a', 'great', 'place', 'to', 'look', 'for', 'food', '.', 'as'] -> he


In [ ]:
emb = xp.random.randn(vocab_size, 32) * 0.1

In [ ]:
class MLP_slm:
    def __init__(self, arc, out):
        self.g = [None] * len(arc)
        for i in range(len(arc)):
            if(i != len(arc) - 1):
                self.g[i] = [xp.random.randn(arc[i + 1], arc[i]) * xp.sqrt(1/arc[i]), xp.zeros((1, arc[i + 1]))]
            else:
                self.g[i] = [xp.random.randn(out, arc[i]) * xp.sqrt(1/arc[i]), xp.zeros((1, out))]
        self.update()
    def forward(self, x):
        for layer in self.layers:
            for module in layer:
                x =  module.forward(x)
        return x.astype(float)
    def update(self):
        self.n = len(self.g)
        self.affine = [None] * self.n
        self.activation = [None] * self.n
        self.layer = [None] * self.n
        self.layers = [None] * self.n

        for i in range(self.n):
            self.affine[i] = Affine(self.g[i][0], self.g[i][1])
            if(i != self.n - 1):
                self.activation[i] = Sigmoid()
            else:
                self.activation[i] = Softmax()
            self.layer[i] = [self.affine[i], self.activation[i]]
            self.layers[i] = (self.layer[i])



    def gd(self, unembX, y_train, lr, emb,B=128):
        idx = xp.random.randint(0, len(unembX), B)
        X_train = emb[unembX[idx]].reshape(B, -1)
        y = xp.eye(vocab_size)[y_train[idx]]
        return self.dgd(X_train, y, lr, emb, unembX[idx])


    def train(self, X_train, y_train, epochs, lr):
        for i in range(epochs):
            self.gd(X_train, y_train, lr)


    def test(self, X_test, y_test):
        preds = xp.argmax(self.forward(X_test), axis=1)
        return float((preds == y_test).mean())



    def dgd(self, x, real, lr, emb, Xb):
        xc = x
        n = len(x)
        ins = []
        outs = [0]
        for layer in self.layers:
            ins.append(x)
            for module in layer:
                x =  module.forward(x)
            outs.append(x)
        diff = x - real

        rolling = diff
        for i in range(self.n - 1, -1, -1):
            self.g[i][1] -= lr / n * rolling.sum(axis=0)
            ww = self.g[i][0].copy()
            self.g[i][0] -= lr / n * (rolling.T @  ins[i])
            rolling = rolling @ ww
            emb -= lr/n * rolling @ ww
            if i > 0:
                rolling *= (outs[i]) * (1 - outs[i])

        dC = xp.zeros_like(emb)
        scatter_add(dC, Xb, rolling.reshape(n, BLOCK_SIZE, 32))
        emb -= lr / n * dC
        return rolling



    def __call__(self, x):
        return self.forward(x)